# Compare Different Models on Real-World Datasets

Evaluate all registered models on OpenML benchmark datasets, with per-model cache reuse and W&B logging.

In [ ]:
from pathlib import Path

from pfns.run_evaluation_cli import (
    build_available_baseline_model_configs,
    compute_per_dataset_stats,
    print_results_summary,
    run_real_world_model_from_config,
    summarize_results,
)
from pfns.experiments.model_benchmarks.hashing import single_model_hash
from pfns.utils import get_default_device
from pfns.experiments.model_benchmarks.io import (
    REAL_WORLD_BUNDLE_KEYS,
    REAL_WORLD_REQUIRED_FILES,
    download_results_bundle_from_wandb,
    find_latest_real_world_bundle_for_model,
    load_dataframe_bundle,
    make_bundle_path,
    make_model_artifact_name,
    
    sanitize_wandb_artifact_component,
    save_dataframe_bundle,
    upload_results_bundle_to_wandb,
)
from pfns.experiments.model_benchmarks.model_registry import (
    MODEL_FAMILIES,
    get_all_models,
    get_baseline_models,
    get_models_from_families,
    get_models_from_names,
)
from pfns.experiments.model_benchmarks.workflows import (
    aggregate_real_world_results_from_bundles,
    alias_real_world_dataframe_bundle,
    build_real_world_run_metadata,
    real_world_bundle_is_compatible,
)

REAL_DATASET_EXPERIMENT = {
    "name": "real_world_openml_comparison",
    "benchmark": "opencc",
    "max_samples": 1000,
    "max_features": 20,
    "max_classes": 10,
    "n_splits": 5,
    "batch_size_inference": 32,
    "n_ensemble_configurations": 32,
    "preprocess_transforms": ["none", "power", "robust"],
    "sample_order_permutation": True,
    "fla_cache_chunk_size": None,
}

BASELINE_EVAL = {
    "n_jobs": 4,
    "random_state": 42,
}

WANDB = {
    "enabled": True,
    "overwrite": False,
    "artifact_name_real_eval": "real_eval_results",
    "entity": "icl_arch",
    "artifact_project": "real_world_eval_artifacts",
    "scores_project": "real_world_eval_scores",
    "mode": "online"
}

OUTPUT_ROOT = Path.cwd().resolve() / "exp_outputs" / "real_world_eval"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Results are stored in: {OUTPUT_ROOT}")
print(f"Available model families: {list(MODEL_FAMILIES)}")


def get_model_combinations(*model_dicts, add_baselines=False):
    combined = {}
    if add_baselines:
        baseline_models_to_compare, skipped_baselines = build_available_baseline_model_configs(
            candidates=get_baseline_models(),
            n_jobs=BASELINE_EVAL["n_jobs"],
            random_state=BASELINE_EVAL["random_state"],
        )
        if skipped_baselines:
            print(f"Skipping unavailable baselines in this environment: {skipped_baselines}")
        combined.update(baseline_models_to_compare)
        
    if not model_dicts:
        return {}
    
    for d in model_dicts:
        combined.update(d)
    return combined

# Model selection
# models_to_compare = get_models_from_families(["transformer", "kda"])
# models_to_compare = get_models_from_names(["Softmax_Transformer"])

# All models
# models_to_compare = get_model_combinations(get_all_models(), add_baselines=True)

# All models without baselines
# models_to_compare = get_model_combinations(get_all_models(), add_baselines=False)

# Only baselines
# models_to_compare = get_model_combinations( {}, add_baselines=True)

# Equal param models
# models_to_compare = get_model_combinations(get_models_from_families(["gla"]))

# Transformer masking expreiments
# models_to_compare = get_model_combinations(get_models_from_families(["transformer_masked"]), add_baselines=False)

# All FLA models
models_to_compare = get_model_combinations(get_models_from_families(["fla_models"]), add_baselines=False)

print(
    "Selected models:", len(models_to_compare)
)

device = get_default_device()
print(f"Using device: {device}")

expected_real_metadata = build_real_world_run_metadata(
    experiment=REAL_DATASET_EXPERIMENT,
    device=device,
)


In [ ]:
if WANDB["enabled"] and WANDB["overwrite"]:
    print("WANDB overwrite=True: skipping per-model download and forcing reruns.")

for model_name, model_config in models_to_compare.items():
    model_hash = single_model_hash(
        model_name=model_name,
        model_config=model_config,
        experiment_payload=REAL_DATASET_EXPERIMENT,
    )
    model_artifact_name = make_model_artifact_name(
        base_artifact_name=WANDB["artifact_name_real_eval"],
        model_name=model_name,
        model_hash=model_hash,
    )

    if WANDB["enabled"] and not WANDB["overwrite"]:
        cached_bundle_path = download_results_bundle_from_wandb(
            artifact_name=model_artifact_name,
            download_root=OUTPUT_ROOT / "wandb_model_cache" / "real_world",
            required_files=REAL_WORLD_REQUIRED_FILES,
            entity=WANDB["entity"],
            project=WANDB["artifact_project"],
        )
        print(f"Checked for cached W&B artifact for {model_name}: {cached_bundle_path}")
        if cached_bundle_path is not None:
            cached_bundle = load_dataframe_bundle(
                cached_bundle_path,
                expected_keys=REAL_WORLD_BUNDLE_KEYS,
            )
            cached_bundle_for_model, source_labels = alias_real_world_dataframe_bundle(
                cached_bundle,
                target_model_name=model_name,
            )
            cached_dataframes = cached_bundle_for_model["dataframes"]

            if real_world_bundle_is_compatible(
                cached_bundle_for_model,
                model_name=model_name,
                expected_metadata=expected_real_metadata,
            ):
                model_bundle_path = make_bundle_path(
                    OUTPUT_ROOT / "real_world",
                    f"{REAL_DATASET_EXPERIMENT['name']}_{sanitize_wandb_artifact_component(model_name)}",
                )
                save_dataframe_bundle(
                    dataframes=cached_dataframes,
                    bundle_dir=model_bundle_path,
                    experiment=REAL_DATASET_EXPERIMENT,
                    run_metadata=expected_real_metadata,
                )
                if source_labels:
                    print(
                        f"Reused cached real-world W&B result for {model_name} from stored labels "
                        f"{sorted(source_labels)}: {cached_bundle_path}. Saved local alias bundle: {model_bundle_path}"
                    )
                else:
                    print(
                        f"Reused cached real-world W&B result for {model_name}: {cached_bundle_path}. "
                        f"Saved local alias bundle: {model_bundle_path}"
                    )
                continue

    print(f"Running real-world benchmark for model: {model_name}")
    results = run_real_world_model_from_config(
        model_config=model_config,
        experiment=REAL_DATASET_EXPERIMENT,
        device=device,
        baseline_n_jobs=BASELINE_EVAL["n_jobs"],
        random_state=BASELINE_EVAL["random_state"],
        verbose=False,
    )

    if not results.empty:
        results["model"] = model_name
    else:
        print(f"Warning: No results for model {model_name}, skipping saving and upload.")
        continue
    summary = summarize_results(results)
    per_dataset = compute_per_dataset_stats(results)

    model_bundle_path = make_bundle_path(
        OUTPUT_ROOT / "real_world",
        f"{REAL_DATASET_EXPERIMENT['name']}_{sanitize_wandb_artifact_component(model_name)}",
    )
    save_dataframe_bundle(
        dataframes={
            "results": results,
            "summary": summary.reset_index() if summary is not None else None,
            "per_dataset": per_dataset,
        },
        bundle_dir=model_bundle_path,
        experiment=REAL_DATASET_EXPERIMENT,
        run_metadata=expected_real_metadata,
    )
    print(f"Saved real-world bundle for {model_name}: {model_bundle_path}")

    if WANDB["enabled"]:
        artifact_ref = upload_results_bundle_to_wandb(
            model_bundle_path,
            artifact_name=model_artifact_name,
            run_name=f"{REAL_DATASET_EXPERIMENT['name']}_{sanitize_wandb_artifact_component(model_name)}_{model_hash}_artifact",
            metadata={
                "experiment": REAL_DATASET_EXPERIMENT,
                "model_name": model_name,
                "model_config": model_config,
                "model_hash": model_hash,
                "run_metadata": expected_real_metadata,
            },
            entity=WANDB["entity"],
            project=WANDB["artifact_project"],
            job_type="real_world_bundle_upload",
        )
        print(f"Uploaded real-world artifact for {model_name}: {artifact_ref}")

print("\nReal-world evaluation completed.")



## Load and aggregate per-model outputs

Collect the latest compatible bundle for each selected model, then build combined result tables.

In [ ]:
from IPython.display import display

from pfns.datasets.tabular_datasets import (
    open_cc_dids as OPENCC_BENCHMARK,
    test_dids_classification as TEST_BENCHMARK,
)


bundles_by_model = {}
missing_models = []

for model_name in models_to_compare:
    bundle_path, bundle = find_latest_real_world_bundle_for_model(
        model_name,
        output_root=OUTPUT_ROOT,
        expected_metadata=expected_real_metadata,
    )
    if bundle is None:
        missing_models.append(model_name)
        continue

    bundles_by_model[model_name] = {"path": bundle_path, "bundle": bundle}
    print(f"Loaded {model_name} bundle: {bundle_path}")

if not bundles_by_model:
    raise RuntimeError(
        "No compatible bundles were found. Run the benchmark cell first, or check metadata settings."
    )

all_results, results_by_model = aggregate_real_world_results_from_bundles(
    bundles_by_model,
    expected_splits=int(REAL_DATASET_EXPERIMENT["n_splits"]),
)

summary = summarize_results(all_results)
per_dataset = compute_per_dataset_stats(all_results)
if summary is None or per_dataset is None or per_dataset.empty:
    raise RuntimeError("Could not compute summary tables from aggregated results.")

benchmark_ids = (
    OPENCC_BENCHMARK
    if REAL_DATASET_EXPERIMENT["benchmark"] == "opencc"
    else TEST_BENCHMARK
)

print("\nAggregated real-world benchmark summary")
print(f"Models loaded: {len(bundles_by_model)} / {len(models_to_compare)}")
print(f"Rows in all_results: {len(all_results)}")
print(f"Datasets covered: {per_dataset['dataset'].nunique()} / {len(benchmark_ids)}")

if missing_models:
    print("Missing compatible bundles for:", missing_models)

print_results_summary(all_results, title="Aggregated Results Across Selected Models")


## Result tables

Display model-level and dataset-level metric tables from the aggregated benchmark results.

In [ ]:
if summary is None or per_dataset is None or summary.empty or per_dataset.empty:
    raise RuntimeError("No summary/per-dataset results available.")

summary_display = summary.copy().sort_values("accuracy_mean", ascending=False)
summary_metric_labels = {
    "accuracy_mean": "Accuracy mean \u2191",
    "accuracy_std": "Accuracy std \u2191",
    "roc_auc_mean": "ROC-AUC mean \u2191",
    "roc_auc_std": "ROC-AUC std \u2191",
    "log_loss_mean": "CE mean \u2193",
    "log_loss_std": "CE std \u2193",
    "ece_mean": "ECE mean \u2193",
    "ece_std": "ECE std \u2193",
}
summary_display = summary_display.rename(columns=summary_metric_labels)
print("Model summary table (mean over datasets):")
display(
    summary_display.style.format(
        {
            "Accuracy mean \u2191": "{:.4f}",
            "Accuracy std \u2191": "{:.4f}",
            "ROC-AUC mean \u2191": "{:.4f}",
            "ROC-AUC std \u2191": "{:.4f}",
            "CE mean \u2193": "{:.4f}",
            "CE std \u2193": "{:.4f}",
            "ECE mean \u2193": "{:.4f}",
            "ECE std \u2193": "{:.4f}",
            "fit_time_mean": "{:.3f}",
            "predict_time_mean": "{:.3f}",
        }
    ).background_gradient(cmap="Greens_r", subset=["Accuracy mean \u2191", "ROC-AUC mean \u2191"])
)

per_dataset_metric_specs = [
    ("accuracy_mean", "Accuracy \u2191"),
    ("roc_auc_mean", "ROC-AUC \u2191"),
    ("log_loss_mean", "CE \u2193"),
    ("ece_mean", "ECE \u2193"),
]
for metric, metric_label in per_dataset_metric_specs:
    print(f"\nPer-dataset table: {metric_label}")
    table = (
        per_dataset
        .pivot_table(index="dataset", columns="model", values=metric, observed=True)
        .sort_index()
    )
    display(table.style.format("{:.4f}").background_gradient(cmap="Blues"))


## Setting Analysis Shared Setup

Shared helpers/constants used by both cross-mode ranking and gain/significance sections.


In [ ]:
from pfns.experiments.model_benchmarks.setting_analysis import (
    SETTING_METRIC_DIRECTION,
    SETTING_METRIC_LABELS,
    get_setting_preprocess,
)
from pfns.experiments.model_benchmarks.comparison_analysis import run_comparison_analysis

## Cross-Mode Setting Ranking

Rank and average `Comb_MT`, `Comb_ST`, `Int_ST`, `Int_MT` across model types using only canonical model names.

Models with extra suffixes (for example `short_conv`, hidden-size sweeps, `Layers_24`, or seq-len variants) are ignored automatically.


In [ ]:
import pandas as pd

if per_dataset is None or per_dataset.empty:
    raise RuntimeError("No per-dataset results are available for setting ranking.")

TARGET_SETTINGS = ["Comb_MT", "Comb_ST", "Int_ST", "Int_MT"]

pre = get_setting_preprocess(
    results_df=per_dataset,
    target_settings=TARGET_SETTINGS,
)

evaluated_df = pre["model_meta"]
evaluated_presence = pre["presence"].reindex(columns=TARGET_SETTINGS, fill_value=False).sort_index()
evaluated_complete_types = pre["eligible_model_types"]
analysis_df = pre["filtered_results"].copy()

print("Canonical setting availability in currently loaded evaluation results.")
display(evaluated_presence)
print(f"Model types with all four settings evaluated: {evaluated_complete_types}")

metric_specs = [
    ("accuracy_mean", True, "Accuracy"),
    ("roc_auc_mean", True, "ROC-AUC"),
    ("log_loss_mean", False, "CE"),
    ("ece_mean", False, "ECE"),
]

missing_metric_cols = [col for col, _, _ in metric_specs if col not in analysis_df.columns]
if missing_metric_cols:
    raise RuntimeError(f"Missing required metric columns for ranking: {missing_metric_cols}")

rank_frames = []
for metric_col, higher_is_better, metric_label in metric_specs:
    metric_rank_df = analysis_df[["dataset", "model_type", "setting", metric_col]].dropna().copy()
    metric_rank_df["rank"] = metric_rank_df.groupby(
        ["model_type", "dataset"], observed=True
    )[metric_col].rank(method="average", ascending=not higher_is_better)
    metric_rank_df["metric"] = metric_label
    rank_frames.append(metric_rank_df[["metric", "setting", "rank"]])

rank_df = pd.concat(rank_frames, ignore_index=True)
rank_summary = (
    rank_df.groupby(["metric", "setting"], observed=True)
    .agg(mean_rank=("rank", "mean"), std_rank=("rank", "std"), n_pairs=("rank", "size"))
    .reset_index()
)

rank_table = (
    rank_summary
    .pivot(index="setting", columns="metric", values="mean_rank")
    .reindex(TARGET_SETTINGS)
)

print("Mean rank by setting across model types and datasets (lower is better).")
display(rank_table.style.format("{:.3f}").background_gradient(cmap="Greens_r", axis=0))

overall_setting_rank = (
    rank_summary.groupby("setting", observed=True)["mean_rank"]
    .mean()
    .reindex(TARGET_SETTINGS)
    .sort_values()
)
print("Overall mean rank across all metrics (lower is better).")
display(overall_setting_rank.to_frame("overall_mean_rank").style.format({"overall_mean_rank": "{:.3f}"}).background_gradient(cmap="Greens_r"))

setting_metric_means = (
    analysis_df.groupby("setting", observed=True)
    .agg(
        accuracy_mean=("accuracy_mean", "mean"),
        roc_auc_mean=("roc_auc_mean", "mean"),
        log_loss_mean=("log_loss_mean", "mean"),
        ece_mean=("ece_mean", "mean"),
    )
    .reindex(TARGET_SETTINGS)
)
print("Raw metric averages by setting across eligible model types.")
display(setting_metric_means.style.format({
    "accuracy_mean": "{:.4f}",
    "roc_auc_mean": "{:.4f}",
    "log_loss_mean": "{:.4f}",
    "ece_mean": "{:.4f}",
}))


## Gain And Significance Across Settings

Paired gain and interval analysis for `Comb_MT`, `Comb_ST`, `Int_MT`, `Int_ST` (not model-vs-model).

Positive gain always means the compared setting is better than the reference setting.


In [ ]:
import matplotlib.pyplot as plt

SETTING_ANALYSIS = {
    "metric": "roc_auc",  # one of: accuracy, roc_auc, log_loss, ece
    "unit": "dataset",  # dataset or split
    "target_labels": ["Comb_MT", "Comb_ST", "Int_MT", "Int_ST"],
    "reference_label": "Int_MT",
    "error_bars": "ci95",  # std or ci95
    "compare_col": "setting",
    "comparison_label": "setting",
    "wilcoxon_alpha": 0.05,
}

if all_results is None or all_results.empty:
    raise RuntimeError("No aggregated all_results dataframe is available for setting CD analysis.")

metric = SETTING_ANALYSIS["metric"]
if metric not in SETTING_METRIC_DIRECTION:
    raise ValueError(f"Unsupported metric {metric!r}. Choose from {list(SETTING_METRIC_DIRECTION)}")
if metric not in all_results.columns:
    raise RuntimeError(f"Metric column {metric!r} is not present in all_results.")

unit = SETTING_ANALYSIS["unit"]
target_labels = list(dict.fromkeys(SETTING_ANALYSIS["target_labels"]))

pre = get_setting_preprocess(
    results_df=all_results,
    target_settings=target_labels,
)
presence = pre["presence"].reindex(columns=target_labels, fill_value=False)
eligible_model_types = pre["eligible_model_types"]
comparison_results = pre["setting_results"]

print("Eligible model types (all requested settings available):", eligible_model_types)
print("Setting availability by model type:")
display(presence)

higher_is_better = SETTING_METRIC_DIRECTION[metric]
pair_cols = ["model_type", "dataset"] if unit == "dataset" else ["model_type", "dataset", "split"]

analysis = run_comparison_analysis(
    comparison_results=comparison_results,
    metric_col=metric,
    metric_label=SETTING_METRIC_LABELS[metric],
    compare_col=SETTING_ANALYSIS["compare_col"],
    target_labels=target_labels,
    pair_cols=pair_cols,
    higher_better=higher_is_better,
    reference_label=SETTING_ANALYSIS["reference_label"],
    unit=unit,
    error_bars=SETTING_ANALYSIS["error_bars"],
    comparison_label=SETTING_ANALYSIS["comparison_label"],
    include_boxplot=True,
    include_pairwise_tables=True,
    include_cd_diagram=True,
    wilcoxon_alpha=SETTING_ANALYSIS["wilcoxon_alpha"],
    empty_error_message=(
        "No complete paired rows found across all requested settings. "
        "Try unit='dataset' or evaluate more models/settings."
    ),
)

print(f"Complete paired rows used: {analysis['n_complete_pairs']}")
print(f"Metric: {SETTING_METRIC_LABELS[metric]} ({metric})")
print(f"Pairing unit: {unit}")
print(f"Reference setting: {analysis['gain']['reference_label']}")
print(f"Comparisons: {analysis['gain']['comparison_labels']}")
print("Setting means:")
display(analysis["gain"]["label_means"].to_frame("mean"))

print("\nPaired gain summary by setting (positive means better than reference setting):")
display(
    analysis["gain"]["gain_summary"].style.format(
        {
            "mean_gain": "{:+.5f}",
            "std_gain": "{:.5f}",
            "sem_gain": "{:.5f}",
            "ci95": "{:.5f}",
            "ci95_low": "{:+.5f}",
            "ci95_high": "{:+.5f}",
            "n_pairs": "{:.0f}",
            "share_pairs_better": "{:.1%}",
        }
    ).background_gradient(subset=["mean_gain"], cmap="RdYlGn")
)

if analysis["pairwise"] is not None:
    print("\nPairwise mean gain matrix (row setting vs column setting, positive means row is better):")
    display(analysis["pairwise"]["pairwise_mean"].style.format("{:+.4f}").background_gradient(cmap="RdYlGn", axis=None))

    print("Pairwise 95% CI half-width matrix:")
    display(analysis["pairwise"]["pairwise_ci95"].style.format("{:.4f}").background_gradient(cmap="Blues", axis=None))

    print("Pairwise significance proxy (95% CI excludes zero):")
    display(analysis["pairwise"]["pairwise_sig"])

for fig_key in ["gain_barh", "gain_boxplot", "wilcoxon_cd"]:
    if fig_key in analysis["figures"]:
        fig, _ = analysis["figures"][fig_key]
        display(fig)
        plt.close(fig)



## Plots

Create metric bars, speed-vs-accuracy tradeoff, and dataset-level rank curves using existing benchmark utilities.

In [ ]:
import matplotlib.pyplot as plt

from pfns.experiments.model_benchmarks.analysis import compute_mean_rank_tables
from pfns.experiments.model_benchmarks.constants import DEFAULT_COLORS
from pfns.experiments.model_benchmarks.plotting import build_model_style_map, plot_curves_from_df

if summary is None or summary.empty:
    raise RuntimeError("No summary dataframe available for plotting.")

ordered_models = summary.sort_values("accuracy_mean", ascending=False).index.tolist()
non_black_colors = [c for c in DEFAULT_COLORS if c.lower() not in {"#000000", "black", "k"}]
style_map = build_model_style_map(ordered_models, colors=non_black_colors)
model_colors = {name: style_map[name][2] for name in ordered_models}
metric_labels = {
    "accuracy": "Accuracy \u2191",
    "roc_auc": "ROC-AUC \u2191",
    "log_loss": "CE \u2193",
    "ece": "ECE \u2193",
}

# 1) Summary metric bar plots
bar_specs = [
    ("accuracy_mean", "accuracy_std", metric_labels["accuracy"], True),
    ("roc_auc_mean", "roc_auc_std", metric_labels["roc_auc"], True),
    ("log_loss_mean", "log_loss_std", metric_labels["log_loss"], False),
    ("ece_mean", "ece_std", metric_labels["ece"], False),
]

fig, axes = plt.subplots(2, 2, figsize=(16, 10), dpi=130)
for ax, (mean_col, std_col, title, higher_is_better) in zip(axes.flat, bar_specs):
    plot_df = summary.reset_index().rename(columns={"index": "model"})
    plot_df = plot_df.sort_values(mean_col, ascending=not higher_is_better)

    std = plot_df[std_col].fillna(0.0)
    ax.barh(
        plot_df["model"],
        plot_df[mean_col],
        xerr=std,
        color=[model_colors[m] for m in plot_df["model"]],
        alpha=0.9,
    )

    # Zoom to observed metric range (+/- std) so differences stay visible.
    lower = float((plot_df[mean_col] - std).min())
    upper = float((plot_df[mean_col] + std).max())
    span = max(upper - lower, 1e-6)
    pad = 0.08 * span
    x_min = lower - pad
    x_max = upper + pad

    if mean_col in {"accuracy_mean", "roc_auc_mean"}:
        x_min = max(0.0, x_min)
        x_max = min(1.0, x_max)
    else:
        x_min = max(0.0, x_min)

    if x_max <= x_min:
        x_max = x_min + max(span, 1e-3)

    ax.set_xlim(x_min, x_max)
    ax.set_title(title)
    ax.grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()

# 2) Accuracy vs inference speed tradeoff
tradeoff_df = summary.reset_index().rename(columns={"index": "model"})
fig, ax = plt.subplots(figsize=(9, 6), dpi=130)
for _, row in tradeoff_df.iterrows():
    model_name = row["model"]
    ax.scatter(
        row["predict_time_mean"],
        row["accuracy_mean"],
        s=90,
        color=model_colors[model_name],
        alpha=0.9,
        label=model_name,
    )
    ax.annotate(model_name, (row["predict_time_mean"], row["accuracy_mean"]), xytext=(4, 4), textcoords="offset points", fontsize=9)

ax.set_xlabel("Predict time mean (s)")
ax.set_ylabel(f"{metric_labels['accuracy']} mean")
ax.set_xscale("log")
ax.grid(alpha=0.25)
ax.set_title(f"{metric_labels['accuracy']} vs Prediction Time (s) \u2193")
plt.show()

# 3) Dataset-wise rank curves with model_benchmarks rank + plotting helpers
rank_base = (
    all_results
    .groupby(["model", "dataset"], observed=True)
    .agg(
        accuracy=("accuracy", "mean"),
        roc_auc=("roc_auc", "mean"),
        log_loss=("log_loss", "mean"),
        ece=("ece", "mean"),
    )
    .reset_index()
)

rank_long = rank_base.melt(
    id_vars=["model", "dataset"],
    value_vars=["accuracy", "roc_auc", "log_loss", "ece"],
    var_name="metric",
    value_name="value",
)
rank_long["metric"] = rank_long["metric"].replace({"accuracy": "acc"})

dataset_to_idx = {
    name: idx
    for idx, name in enumerate(sorted(rank_long["dataset"].unique()), start=1)
}
rank_input = rank_long.assign(seqlen=rank_long["dataset"].map(dataset_to_idx))
rank_input["rep"] = rank_input["dataset"]

rank_tables = compute_mean_rank_tables(
    rank_input,
    x_col="seqlen",
    higher_is_better_metrics={"acc", "roc_auc"},
)

overall_ranks = rank_tables["overall_ranks"].copy()
overall_ranks["metric"] = overall_ranks["metric"].replace({"acc": "accuracy"})
rank_metric_labels = {
    "accuracy": "Accuracy Rank \u2193",
    "roc_auc": "ROC-AUC Rank \u2193",
    "log_loss": "CE Rank \u2193",
    "ece": "ECE Rank \u2193",
}
print("Overall mean rank (lower is better \u2193):")
for metric in ["accuracy", "roc_auc", "log_loss", "ece"]:
    ranked = (
        overall_ranks[overall_ranks["metric"] == metric]
        .sort_values("rank")
        .drop(columns="metric")
        .reset_index(drop=True)
    )
    print(f"\n{rank_metric_labels[metric]}")
    display(ranked.style.format({"rank": "{:.2f}"}).background_gradient(subset=["rank"], cmap="Greens_r"))

rank_curve_df = rank_tables["x_ranks"].copy()
rank_curve_df["metric"] = rank_curve_df["metric"].replace(
    {
        "acc": "accuracy_rank",
        "roc_auc": "roc_auc_rank",
        "log_loss": "log_loss_rank",
        "ece": "ece_rank",
    }
)

plot_curves_from_df(
    rank_curve_df,
    specs=[
        ("accuracy_rank", "Accuracy Rank \u2193"),
        ("roc_auc_rank", "ROC-AUC Rank \u2193"),
        ("log_loss_rank", "CE Rank \u2193"),
        ("ece_rank", "ECE Rank \u2193"),
    ],
    style_map=style_map,
    x_col="seqlen",
    value_col="rank",
    x_label="Dataset index (sorted by dataset name)",
    title_suffix=" (1 is best)",
    invert_y=True,
    show_std=False,
    figsize=(24, 7),
)
